# AML 3303 — Hands-On: A Mini ETL Pipeline with Quality Checks (30 min)

## Scenario
**ShopNorth**, an online retailer, exports its customer support tickets as a JSON file every day. The
operations team wants them in a database they can query, but the exports are messy: inconsistent
categories, duplicates, broken dates, missing emails, and customers' phone numbers and emails sitting
inside the message text.

You will build the pipeline from the slides, end to end:

```
Extract ─▶ Profile ─▶ Standardise ─▶ Redact PII ─▶ Validate ─┬─▶ Load (tickets) ─▶ Query
                                                             └─▶ Quarantine (bad rows)
                         + a run log every time, + an incremental load on day 2
```

Only `pandas` and Python's built-in `sqlite3` are needed. Nothing to install.

Fill in every cell marked `# TODO`. Hints sit directly above each one.

## 0. Setup

Run once. Do not change it.

In [ ]:
# Run this cell once. It creates two daily exports from the ticketing system.
import json, re, time, sqlite3
import pandas as pd

day1 = [
    {"ticket_id": 1001, "customer_email": "priya.s@example.com", "category": "Billing",  "priority": "high",   "created_at": "2026-10-01 09:12", "updated_at": "2026-10-01 09:12", "message": "I was charged twice. Call me at 416-555-0199."},
    {"ticket_id": 1002, "customer_email": "liam.b@example.com",  "category": "shipping", "priority": "Medium", "created_at": "2026-10-01 10:05", "updated_at": "2026-10-01 10:05", "message": "Order hasn't arrived after 10 days."},
    {"ticket_id": 1003, "customer_email": "ana.r@example",       "category": " Returns ", "priority": "low",   "created_at": "2026-10-01 11:40", "updated_at": "2026-10-01 11:40", "message": "How do I return a jacket? Email me at ana.r@example.com"},
    {"ticket_id": 1004, "customer_email": "omar.k@example.com",  "category": "BILLING",  "priority": "urgent", "created_at": "2026-10-01 12:02", "updated_at": "2026-10-01 12:02", "message": "Card ending 4242 declined but money taken."},
    {"ticket_id": 1002, "customer_email": "liam.b@example.com",  "category": "shipping", "priority": "Medium", "created_at": "2026-10-01 10:05", "updated_at": "2026-10-01 10:05", "message": "Order hasn't arrived after 10 days."},
    {"ticket_id": 1005, "customer_email": None,                  "category": "Shipping", "priority": "high",   "created_at": "2026-10-01 14:30", "updated_at": "2026-10-01 14:30", "message": "Package damaged, phone (647) 555-0123."},
    {"ticket_id": 1006, "customer_email": "mei.c@example.com",   "category": "returns",  "priority": "low",    "created_at": "not a date",       "updated_at": "2026-10-01 15:10", "message": "Wrong size sent."},
    {"ticket_id": 1007, "customer_email": "sam.t@example.com",   "category": "Account",  "priority": "medium", "created_at": "2026-10-01 16:45", "updated_at": "2026-10-01 16:45", "message": "Can't reset my password."},
]

day2 = [
    {"ticket_id": 1002, "customer_email": "liam.b@example.com",  "category": "Shipping", "priority": "high",   "created_at": "2026-10-01 10:05", "updated_at": "2026-10-02 08:30", "message": "Still no package. Escalating, call 905-555-0142."},
    {"ticket_id": 1008, "customer_email": "zoe.l@example.com",   "category": "billing",  "priority": "medium", "created_at": "2026-10-02 09:15", "updated_at": "2026-10-02 09:15", "message": "Need an invoice copy for my company."},
    {"ticket_id": 1009, "customer_email": "raj.p@example.com",   "category": "Account",  "priority": "low",    "created_at": "2026-10-02 11:00", "updated_at": "2026-10-02 11:00", "message": "Update my address please."},
    {"ticket_id": 1001, "customer_email": "priya.s@example.com", "category": "Billing",  "priority": "high",   "created_at": "2026-10-01 09:12", "updated_at": "2026-10-01 09:12", "message": "I was charged twice. Call me at 416-555-0199."},
]

json.dump(day1, open("tickets_2026-10-01.json", "w"), indent=1)
json.dump(day2, open("tickets_2026-10-02.json", "w"), indent=1)
print("Exports written: tickets_2026-10-01.json, tickets_2026-10-02.json")

## 1. Extract (3 min)

Read the day-1 export into a DataFrame.

**Hints:** `pd.read_json("file.json")` or `pd.DataFrame(json.load(open(...)))`

In [ ]:
# TODO: load tickets_2026-10-01.json into raw_df and display it
raw_df = ...
raw_df

## 2. Profile the data (4 min)

Before changing anything, find out what is wrong. Real exports are messy.

**Hints**
- `df.duplicated().sum()`
- `df.isna().sum()`
- `df["category"].unique()` shows inconsistent spellings
- `pd.to_datetime(df["created_at"], errors="coerce").isna()` finds unparseable dates

In [ ]:
# TODO: print duplicates, missing values, the distinct categories and priorities,
#       and how many created_at values cannot be parsed as dates


## 3. Transform: standardise (6 min)

Write `standardise(df)` so every later step can rely on clean values.

**Hints**
- Drop exact duplicates: `df.drop_duplicates()`
- Categories: `.str.strip().str.title()` turns `" Returns "` and `"BILLING"` into `"Returns"` and `"Billing"`
- Priority: `.str.lower()`
- Dates: `pd.to_datetime(..., errors="coerce")` turns bad values into `NaT` instead of crashing

In [ ]:
def standardise(df):
    df = df.copy()
    # TODO: drop duplicate rows
    # TODO: clean category and priority
    # TODO: convert created_at and updated_at to datetimes (bad values -> NaT)
    return df

clean_df = standardise(raw_df)
clean_df

## 4. Transform: redact PII (5 min)

Ticket messages contain phone numbers and email addresses. Anything sent onward, to a dashboard or an
LLM, should not carry them. This is the governance step from the slides, and under PIPEDA it is not optional.

**Hints**
- Phone numbers like `416-555-0199` and `(647) 555-0123`: `r"\(?\d{3}\)?[\s-]?\d{3}-\d{4}"`
- Emails: `r"[\w.+-]+@[\w-]+\.[\w.]+"`
- `re.sub(pattern, "[PHONE]", text)`

In [ ]:
PHONE = r"..."   # TODO
EMAIL = r"..."   # TODO

def redact(text):
    # TODO: replace phones with [PHONE] and emails with [EMAIL]
    return text

clean_df["message"] = clean_df["message"].apply(redact)
clean_df[["ticket_id", "message"]]

## 5. Validate and quarantine (5 min)

Some rows cannot be fixed automatically. Do not delete them silently and do not load them either.
Move them to a **quarantine** table with the reason, so someone can review them.

Rules for a valid ticket:
1. `customer_email` is present and looks like an email (contains `@` and a `.` after it)
2. `created_at` is a real date
3. `priority` is one of `low`, `medium`, `high`, `urgent`

**Hints**
- `df["customer_email"].str.contains(r"@.+\.", na=False)`
- `df["created_at"].notna()`
- `df["priority"].isin([...])`

In [ ]:
VALID_PRIORITIES = ["low", "medium", "high", "urgent"]

def validate(df):
    # TODO: build one boolean Series per rule, then combine them
    ok = ...
    reasons = ...   # optional: a text column saying which rule failed
    return df[ok], df[~ok]

good_df, quarantine_df = validate(clean_df)
print(len(good_df), "valid,", len(quarantine_df), "quarantined")
quarantine_df

## 6. Load into SQLite and log the run (4 min)

SQLite is a full SQL database in a single file, so it stands in for the warehouse.

Every run also writes a row to a `pipeline_runs` table: when it ran, rows in, rows loaded, rows rejected.
That is the monitoring idea from the slides, and the same pattern as logging LLM calls.

**Hints**
- `df.to_sql("tickets", conn, if_exists="replace", index=False)`
- Convert datetimes to text first so SQLite stores them cleanly: `.dt.strftime("%Y-%m-%d %H:%M")`

In [ ]:
conn = sqlite3.connect("support.db")

def load(df, table, conn, mode="replace"):
    out = df.copy()
    for col in ["created_at", "updated_at"]:
        out[col] = out[col].dt.strftime("%Y-%m-%d %H:%M")
    # TODO: write `out` to `table` using to_sql
    ...

def log_run(conn, source, rows_in, loaded, rejected, seconds):
    # TODO: append one row to a pipeline_runs table
    ...

start = time.time()
load(good_df, "tickets", conn)
load(quarantine_df, "quarantine", conn)
log_run(conn, "tickets_2026-10-01.json", len(raw_df), len(good_df), len(quarantine_df), time.time() - start)

pd.read_sql("SELECT * FROM pipeline_runs", conn)

## 7. Incremental load: day 2 (5 min)

Day 2's export has a new ticket, an **updated** ticket (1002 was escalated) and an **old** row that was
already loaded (1001). Reloading everything every day does not scale. Instead, load only what changed.
This is the idea behind CDC.

1. Find the **watermark**: the latest `updated_at` already in the warehouse.
2. Keep only day-2 rows with `updated_at` after the watermark.
3. **Upsert:** delete the old versions of those ticket IDs, then insert the new ones.

**Hints**
- `pd.read_sql("SELECT MAX(updated_at) AS wm FROM tickets", conn)`
- Reuse `standardise`, `redact` and `validate` from above. That is why they are functions.
- `conn.execute(f"DELETE FROM tickets WHERE ticket_id IN ({ids})")`

In [ ]:
watermark = ...   # TODO: latest updated_at in the tickets table, as a Timestamp
print("watermark:", watermark)

day2_raw = pd.read_json("tickets_2026-10-02.json", dtype={"created_at": str, "updated_at": str})
day2 = standardise(day2_raw)
day2["message"] = day2["message"].apply(redact)

changed = ...     # TODO: rows with updated_at after the watermark
print(len(changed), "of", len(day2), "rows changed")

good2, bad2 = validate(changed)
# TODO: delete old versions of good2's ticket_ids, then append good2


pd.read_sql("SELECT ticket_id, category, priority, updated_at, message FROM tickets ORDER BY ticket_id", conn)

## 8. Use the data (2 min)

The operations team wants open tickets per category, urgent and high first.

**Hint:** `GROUP BY category`, and `SUM(priority IN ('high','urgent'))` counts the high-priority ones.

In [ ]:
# TODO: one SQL query: tickets per category, with a count of high/urgent ones
